# Classificação de Doenças Respiratórias a Partir de Sintomas
# Utilizando Técnicas de Aprendizagem de Máquina

**Disciplina:** Sistemas de Informação - Aprendizagem de Máquina  
**Professor:** Bruno Rafael Araújo Vasconcelos - UNIFACISA  
**Dataset:** [Diseases and Symptoms - Kaggle](https://www.kaggle.com/datasets/dhivyeshrk/diseases-and-symptoms-dataset)  
**Ambiente:** Jupyter Notebook com Python 3, Scikit-learn, Pandas, Matplotlib e Seaborn

---

## Resumo

Este trabalho apresenta o desenvolvimento e a avaliação de modelos de aprendizagem de máquina para a classificação de três doenças respiratórias (**Pneumonia**, **Bronquite Aguda** e **Asma**) a partir de sintomas binários do dataset real do Kaggle (246.945 registros, 773 doenças). Foram comparados 4 algoritmos: Árvore de Decisão, Random Forest, Regressão Logística e Rede Neural (MLP). Todos alcançaram acurácia próxima de 90%, demonstrando viabilidade na classificação.

**Palavras-chave:** aprendizagem de máquina, classificação de doenças respiratórias, seleção de features, árvore de decisão, random forest, validação cruzada.

## 1. Importação das Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay
)
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Configuração visual
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12})
print('Bibliotecas importadas com sucesso!')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Scikit-learn: {__import__("sklearn").__version__}')

## 2. Carregamento e Filtragem do Dataset

O dataset original do Kaggle contém **246.945 registros** com **773 doenças** e **377 sintomas** binários.  
As três doenças escolhidas são:
- **Pneumonia** — infecção pulmonar com sintomas sistêmicos
- **Bronquite Aguda (Acute Bronchitis)** — inflamação dos brônquios
- **Asma (Asthma)** — condição alérgica crônica

**Justificativa da escolha:** são doenças respiratórias com sobreposição de sintomas, tornando a tarefa de classificação desafiadora.

In [ ]:
# Carregar o dataset completo do Kaggle
df_full = pd.read_csv('Final_Augmented_dataset_Diseases_and_Symptoms.csv')
print(f'Dataset original: {df_full.shape[0]:,} amostras x {df_full.shape[1]} colunas')
print(f'Doenças únicas: {df_full["diseases"].nunique()}')

# Filtrar as 3 doenças escolhidas
DOENÇAS_ESCOLHIDAS = ['pneumonia', 'acute bronchitis', 'asthma']
df = df_full[df_full['diseases'].isin(DOENÇAS_ESCOLHIDAS)].copy().reset_index(drop=True)
print(f'\nDataset filtrado (3 doenças): {df.shape[0]:,} amostras x {df.shape[1]} colunas')
print(f'\nDistribuição das classes:')
print(df['diseases'].value_counts())

# Remover colunas com variância zero (sintomas irrelevantes)
X = df.drop('diseases', axis=1)
colunas_relevantes = X.columns[X.sum() > 0].tolist()
colunas_removidas = X.columns[X.sum() == 0].tolist()
df = df[['diseases'] + colunas_relevantes]

print(f'\nSintomas com variância > 0: {len(colunas_relevantes)}')
print(f'Sintomas removidos (todos zeros): {len(colunas_removidas)}')
print(f'Dataset limpo: {df.shape}')
print(f'Valores nulos: {df.isnull().sum().sum()}')
print(f'\nSintomas relevantes:')
for i, s in enumerate(colunas_relevantes):
    print(f'  {i+1:2d}. {s}')

## 3. Análise Exploratória dos Dados (EDA)

Etapa fundamental para compreender a estrutura dos dados e identificar padrões que orientem a modelagem.

In [ ]:
# 3.1 Distribuição das classes
fig, ax = plt.subplots(figsize=(8, 5))
cores = ['#2196F3', '#FF5722', '#4CAF50']
contagem = df['diseases'].value_counts()
contagem.plot(kind='bar', color=cores, ax=ax, edgecolor='black')
ax.set_title('Distribuição das Doenças no Dataset (Kaggle)')
ax.set_xlabel('Doença'); ax.set_ylabel('Quantidade')
for i, v in enumerate(contagem.values):
    ax.text(i, v+15, str(v), ha='center', fontweight='bold')
ax.set_xticklabels([d.title() for d in contagem.index], rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Frequência dos sintomas por doença
X = df.drop('diseases', axis=1)
frequencia_por_doenca = df.groupby('diseases')[X.columns.tolist()].mean()
variancia_entre_classes = frequencia_por_doenca.var(axis=0).sort_values(ascending=False)
sintomas_ordenados = variancia_entre_classes.index.tolist()

fig, ax = plt.subplots(figsize=(14, 8))
freq_plot = frequencia_por_doenca[sintomas_ordenados].T
freq_plot.columns = [c.title() for c in freq_plot.columns]
freq_plot.plot(kind='barh', ax=ax, color=cores, edgecolor='black', alpha=0.85)
ax.set_title('Frequência dos Sintomas por Doença (Ordenado por Poder Discriminativo)')
ax.set_xlabel('Proporção de Pacientes com o Sintoma')
ax.legend(title='Doença', loc='lower right')
plt.tight_layout()
plt.show()

print('Ranking de sintomas por poder discriminativo (variância entre classes):')
for i, (s, v) in enumerate(variancia_entre_classes.items()):
    print(f'  {i+1:2d}. {s:35s} variância = {v:.4f}')

In [ ]:
# 3.3 Heatmap de correlação entre sintomas
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df[sintomas_ordenados].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlação entre Sintomas Relevantes')
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Total de sintomas por paciente por doença
df_temp = df.copy()
df_temp['total_sintomas'] = X.sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
for i, doenca in enumerate(sorted(df['diseases'].unique())):
    subset = df_temp[df_temp['diseases'] == doenca]['total_sintomas']
    ax.hist(subset, bins=15, alpha=0.6, label=doenca.title(), color=cores[i], edgecolor='black')
ax.set_title('Distribuição do Número Total de Sintomas por Doença')
ax.set_xlabel('Número de Sintomas Presentes'); ax.set_ylabel('Frequência')
ax.legend()
plt.tight_layout()
plt.show()

print('Estatísticas do total de sintomas por doença:')
print(df_temp.groupby('diseases')['total_sintomas'].describe().round(2))

## 4. Seleção de Features (Feature Selection)

A seleção de features é essencial para:
- **Reduzir o overfitting** ao eliminar variáveis irrelevantes
- **Diminuir o tempo de treinamento** dos modelos
- **Melhorar a interpretabilidade** dos resultados

Técnicas utilizadas: **Teste Chi-Quadrado** e **Informação Mútua**.

In [ ]:
# Codificar variável-alvo
le = LabelEncoder()
y_codificado = le.fit_transform(df['diseases'])
classes = le.classes_
classes_titulo = [c.title() for c in classes]
print(f'Classes: {dict(zip(range(len(classes)), classes_titulo))}')

# 4.1 Teste Chi-Quadrado
seletor_chi2 = SelectKBest(chi2, k='all')
seletor_chi2.fit(X, y_codificado)
scores_chi2 = pd.Series(seletor_chi2.scores_, index=X.columns).sort_values(ascending=False)

# 4.2 Informação Mútua
scores_mi = pd.Series(
    mutual_info_classif(X, y_codificado, random_state=42),
    index=X.columns
).sort_values(ascending=False)

# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
scores_chi2.plot(kind='barh', color='#2196F3', edgecolor='black', ax=axes[0])
axes[0].set_title('Seleção de Features - Teste Chi-Quadrado')
axes[0].set_xlabel('Score Chi²'); axes[0].invert_yaxis()
scores_mi.plot(kind='barh', color='#FF5722', edgecolor='black', ax=axes[1])
axes[1].set_title('Seleção de Features - Informação Mútua')
axes[1].set_xlabel('Score MI'); axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

# Tabela comparativa
print('\nComparação dos rankings de features:')
comparacao = pd.DataFrame({'Chi²': scores_chi2, 'Info. Mútua': scores_mi})
print(comparacao.round(4).to_string())

In [ ]:
# 4.3 Impacto do número de features na acurácia
n_total_features = X.shape[1]
k_valores = list(range(3, n_total_features + 1))
acuracias = []

for k in k_valores:
    sel_k = SelectKBest(chi2, k=k)
    X_k = sel_k.fit_transform(X, y_codificado)
    Xtr, Xte, ytr, yte = train_test_split(X_k, y_codificado, test_size=0.3,
                                           random_state=42, stratify=y_codificado)
    clf = DecisionTreeClassifier(max_depth=10, random_state=42)
    clf.fit(Xtr, ytr)
    acuracias.append(accuracy_score(yte, clf.predict(Xte)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_valores, acuracias, 'o-', color='#4CAF50', linewidth=2)
ax.set_xlabel('Número de Features Selecionadas')
ax.set_ylabel('Acurácia no Conjunto de Teste')
ax.set_title('Impacto do Número de Features na Acurácia do Modelo')
ax.axhline(y=max(acuracias), color='red', linestyle=':', alpha=0.5,
           label=f'Máximo: {max(acuracias):.4f}')
ax.legend()
plt.tight_layout()
plt.show()

# Usar todas as 17 features (poucas e todas relevantes)
features_selecionadas = X.columns.tolist()
print(f'Total de features disponíveis: {n_total_features}')
print(f'Features selecionadas: {len(features_selecionadas)} (todas com variância > 0)')
print(f'Acurácia estabiliza em ~9 features (os 9 sintomas discriminativos)')

## 5. Treinamento e Avaliação dos Modelos

Quatro modelos foram implementados e comparados:
- **Árvore de Decisão** — modelo interpretável, bom para dados binários
- **Random Forest** — ensemble de árvores para maior robustez
- **Regressão Logística** — modelo linear robusto e estável
- **Rede Neural (MLP)** — modelo não-linear com camadas ocultas

**Divisão dos dados:** 70% treino (2.330 amostras) / 30% teste (999 amostras) com estratificação.

In [ ]:
# Preparar dados
X_final = df[features_selecionadas]
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X_final, y_codificado, test_size=0.3, random_state=42, stratify=y_codificado
)
print(f'Conjunto de treino: {X_treino.shape[0]} amostras')
print(f'Conjunto de teste:  {X_teste.shape[0]} amostras')
print(f'Features utilizadas: {len(features_selecionadas)}')

# Definir modelos
modelos = {
    'Árvore de Decisão': DecisionTreeClassifier(
        max_depth=10, random_state=42, min_samples_leaf=5
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=15, random_state=42, n_jobs=-1
    ),
    'Regressão Logística': LogisticRegression(
        max_iter=1000, random_state=42, C=1.0
    ),
    'Rede Neural (MLP)': MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=500,
        random_state=42, early_stopping=True
    ),
}

# Validação cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Treinar e avaliar cada modelo
resultados = {}
for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    y_pred = modelo.predict(X_teste)
    cv_scores = cross_val_score(modelo, X_final, y_codificado, cv=cv, scoring='accuracy')
    
    resultados[nome] = {
        'modelo': modelo, 'y_pred': y_pred,
        'acuracia': accuracy_score(y_teste, y_pred),
        'precisao': precision_score(y_teste, y_pred, average='weighted'),
        'recall': recall_score(y_teste, y_pred, average='weighted'),
        'f1': f1_score(y_teste, y_pred, average='weighted'),
        'cv_media': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'matriz_confusao': confusion_matrix(y_teste, y_pred),
    }
    
    print(f'\n{"=" * 50}')
    print(f'{nome}')
    print(f'{"=" * 50}')
    print(f'  Acurácia:  {resultados[nome]["acuracia"]:.4f}')
    print(f'  Precisão:  {resultados[nome]["precisao"]:.4f}')
    print(f'  Recall:    {resultados[nome]["recall"]:.4f}')
    print(f'  F1-Score:  {resultados[nome]["f1"]:.4f}')
    print(f'  CV 5-fold: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'\n{classification_report(y_teste, y_pred, target_names=classes_titulo)}')

## 6. Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, (nome, res) in zip(axes.flatten(), resultados.items()):
    ConfusionMatrixDisplay(
        res['matriz_confusao'], display_labels=classes_titulo
    ).plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'Matriz de Confusão - {nome}')
plt.tight_layout()
plt.show()

## 7. Comparação de Métricas entre Modelos

In [ ]:
# Tabela consolidada de métricas
tabela_metricas = pd.DataFrame({
    nome: {
        'Acurácia': r['acuracia'],
        'Precisão': r['precisao'],
        'Recall': r['recall'],
        'F1-Score': r['f1'],
        'CV Média': r['cv_media'],
    }
    for nome, r in resultados.items()
}).T

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(tabela_metricas))
largura = 0.15
cores_metricas = ['#2196F3', '#FF5722', '#4CAF50', '#FFC107', '#9C27B0']
for i, col in enumerate(tabela_metricas.columns):
    ax.bar(x + i * largura, tabela_metricas[col], largura,
           label=col, color=cores_metricas[i], edgecolor='black', alpha=0.85)
ax.set_xticks(x + largura * 2)
ax.set_xticklabels(tabela_metricas.index, rotation=15, ha='right')
ax.set_ylabel('Score'); ax.set_title('Comparação de Métricas entre Modelos')
ax.legend(loc='lower right')
ax.set_ylim(max(0, tabela_metricas.min().min() - 0.1), 1.05)
plt.tight_layout()
plt.show()

print('Tabela de Resultados Consolidados:')
print(tabela_metricas.round(4).to_string())
print(f'\nMelhor modelo (F1-Score): {tabela_metricas["F1-Score"].idxmax()}')

In [ ]:
# Validação Cruzada - Boxplot
fig, ax = plt.subplots(figsize=(10, 6))
dados_cv = [cross_val_score(m, X_final, y_codificado, cv=cv, scoring='accuracy')
            for m in modelos.values()]
bp = ax.boxplot(dados_cv, labels=list(modelos.keys()), patch_artist=True)
cores_box = ['#2196F3', '#FF5722', '#4CAF50', '#FFC107']
for patch, cor in zip(bp['boxes'], cores_box):
    patch.set_facecolor(cor); patch.set_alpha(0.7)
ax.set_ylabel('Acurácia')
ax.set_title('Validação Cruzada Estratificada (5-fold) - Comparação entre Modelos')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## 8. Interpretabilidade dos Modelos

In [ ]:
# 8.1 Visualização da Árvore de Decisão (primeiros 3 níveis)
fig, ax = plt.subplots(figsize=(22, 12))
plot_tree(
    resultados['Árvore de Decisão']['modelo'],
    feature_names=features_selecionadas,
    class_names=classes_titulo,
    filled=True, rounded=True, ax=ax,
    max_depth=3, fontsize=10
)
ax.set_title('Árvore de Decisão (Primeiros 3 Níveis)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 8.2 Importância das Features - Random Forest
rf = resultados['Random Forest']['modelo']
importancias = pd.Series(
    rf.feature_importances_, index=features_selecionadas
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importancias.plot(kind='barh', color='#4CAF50', edgecolor='black', ax=ax)
ax.set_title('Importância das Features - Random Forest')
ax.set_xlabel('Importância (Gini)')
plt.tight_layout()
plt.show()

print('Ranking de importância das features (Random Forest):')
for i, (feat, imp) in enumerate(importancias.sort_values(ascending=False).items()):
    print(f'  {i+1:2d}. {feat:35s} {imp:.4f}')

In [ ]:
# 8.3 Métricas por classe do melhor modelo
melhor_nome = tabela_metricas['F1-Score'].idxmax()
y_pred_melhor = resultados[melhor_nome]['y_pred']

prec_classe = precision_score(y_teste, y_pred_melhor, average=None)
rec_classe = recall_score(y_teste, y_pred_melhor, average=None)
f1_classe = f1_score(y_teste, y_pred_melhor, average=None)

fig, ax = plt.subplots(figsize=(10, 6))
x_c = np.arange(len(classes_titulo)); w = 0.25
ax.bar(x_c - w, prec_classe, w, label='Precisão', color='#2196F3', edgecolor='black')
ax.bar(x_c, rec_classe, w, label='Recall', color='#FF5722', edgecolor='black')
ax.bar(x_c + w, f1_classe, w, label='F1-Score', color='#4CAF50', edgecolor='black')
ax.set_xticks(x_c); ax.set_xticklabels(classes_titulo)
ax.set_ylabel('Score'); ax.set_title(f'Métricas por Classe - {melhor_nome}')
ax.legend(); ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

print(f'\nMétricas detalhadas por classe ({melhor_nome}):')
for i, c in enumerate(classes_titulo):
    print(f'  {c:20s} Precisão={prec_classe[i]:.3f}  Recall={rec_classe[i]:.3f}  F1={f1_classe[i]:.3f}')

## 9. Conclusão

### Resultados Consolidados

| Modelo | Acurácia | F1-Score | CV (5-fold) |
|--------|----------|----------|-------------|
| **Árvore de Decisão** | **89,7%** | **89,7%** | 90,3% |
| Random Forest | 88,9% | 88,9% | 88,8% |
| Regressão Logística | 89,6% | 89,6% | **90,3%** |
| Rede Neural (MLP) | 89,7% | 89,6% | 90,3% |

### Principais Conclusões

1. **Dataset real do Kaggle**: 246.945 amostras filtradas para 3.329 (3 doenças respiratórias)
2. **Redução dimensional expressiva**: de 377 sintomas originais, apenas **17 são relevantes** para estas doenças e **9 são verdadeiramente discriminativos**
3. **Todos os modelos alcançaram ~90% de acurácia**, demonstrando viabilidade da classificação por sintomas
4. **Sintomas mais discriminativos**: reação alérgica (separa Asma), expectoração (separa Bronquite), sintomas sistêmicos como vômito e calafrios (separam Pneumonia)
5. **Recomendação prática**: Árvore de Decisão pela interpretabilidade ou Regressão Logística pela estabilidade na validação cruzada
6. O principal desafio está na **confusão entre Pneumonia e Bronquite Aguda**, que compartilham muitos sintomas respiratórios comuns
7. O limite de performance (~90%) está na **natureza dos dados** (sobreposição de sintomas), não na capacidade dos algoritmos

### Trabalhos Futuros

- Incluir variáveis numéricas (temperatura, frequência cardíaca)
- Aplicar Grid Search para otimização de hiperparâmetros
- Validar com dados clínicos reais em ambiente hospitalar
- Expandir para mais doenças respiratórias

---
*Todo o trabalho foi desenvolvido em Jupyter Notebook utilizando Python, Scikit-learn, Pandas, Matplotlib e Seaborn.*